# Level 3 — Task 3: Neural Networks with TensorFlow/Keras
**Dataset:** Churn (bigml-20 + bigml-80 merged) | **Tools:** TensorFlow/Keras, pandas, scikit-learn, matplotlib

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve)

print(f"TensorFlow version: {tf.__version__}")
sns.set_theme(style="whitegrid")
tf.random.set_seed(42)
np.random.seed(42)


## 2. Load & Merge Dataset

In [ ]:
from google.colab import files
print("Upload both churn-bigml-80.csv and churn-bigml-20.csv")
uploaded = files.upload()

dfs = [pd.read_csv(f) for f in uploaded.keys()]
df = pd.concat(dfs, ignore_index=True)

print(f"Merged shape : {df.shape}")
print(f"Churn balance:")
print(df["Churn"].value_counts())
df.head()


## 3. Preprocessing

In [ ]:
df_proc = df.copy()
df_proc.drop(columns=["State", "Area code"], inplace=True)

# Encode binary categorical columns
binary_cols = ["International plan", "Voice mail plan"]
le = LabelEncoder()
for col in binary_cols:
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))

# Encode target
df_proc["Churn"] = (df_proc["Churn"] == True) | (df_proc["Churn"] == "True")
df_proc["Churn"] = df_proc["Churn"].astype(int)

assert df_proc.isnull().sum().sum() == 0
print("No missing values. Shape:", df_proc.shape)
print(f"Class balance — Churn=1: {df_proc['Churn'].mean():.2%}")


In [ ]:
X = df_proc.drop(columns=["Churn"]).values
y = df_proc["Churn"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train : {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Input features: {X_train.shape[1]}")


## 4. Build Neural Network Architecture

In [ ]:
def build_model(input_dim, dropout_rate=0.3, l2_lambda=0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_lambda)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),

        layers.Dense(64, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_lambda)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),

        layers.Dense(32, activation="relu"),
        layers.Dropout(dropout_rate / 2),

        layers.Dense(1, activation="sigmoid")
    ])
    return model

model = build_model(X_train.shape[1])
model.summary()


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)


## 5. Train the Model

In [ ]:
# Class weights to handle imbalance
neg, pos = np.bincount(y_train)
class_weight = {0: 1.0, 1: neg / pos}
print(f"Class weights: {class_weight}")

cb_list = [
    callbacks.EarlyStopping(monitor="val_auc", patience=15,
                            restore_best_weights=True, mode="max"),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                patience=5, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=64,
    class_weight=class_weight,
    callbacks=cb_list,
    verbose=1
)

print(f"\nTraining stopped at epoch: {len(history.history['loss'])}")


## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
hist = history.history

for ax, metric, title in zip(
    axes,
    [("loss", "val_loss"), ("accuracy", "val_accuracy"), ("auc", "val_auc")],
    ["Loss", "Accuracy", "AUC"]
):
    ax.plot(hist[metric[0]], label="Train", linewidth=2)
    ax.plot(hist[metric[1]], label="Validation", linewidth=2, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()

plt.suptitle("Training & Validation Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Evaluate on Test Set

In [ ]:
test_loss, test_acc, test_auc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test AUC      : {test_auc:.4f}")


In [ ]:
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Not Churned", "Churned"]))


## 8. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Not Churned", "Churned"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title("Confusion Matrix")

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_val = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, linewidth=2, color="#4C72B0", label=f"AUC = {auc_val:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.suptitle("Neural Network — Model Evaluation", fontweight="bold")
plt.tight_layout()
plt.show()


## 9. Threshold Tuning

In [ ]:
# Find optimal threshold by maximizing F1
from sklearn.metrics import f1_score as f1

thresholds_range = np.arange(0.1, 0.9, 0.01)
f1_scores = [f1(y_test, (y_prob >= t).astype(int)) for t in thresholds_range]
optimal_threshold = thresholds_range[np.argmax(f1_scores)]

print(f"Optimal threshold (max F1): {optimal_threshold:.2f}")

y_pred_tuned = (y_prob >= optimal_threshold).astype(int)
print("\nClassification Report (tuned threshold):")
print(classification_report(y_test, y_pred_tuned, target_names=["Not Churned", "Churned"]))

plt.figure(figsize=(9, 4))
plt.plot(thresholds_range, f1_scores, color="#4C72B0", linewidth=2)
plt.axvline(optimal_threshold, color="red", linestyle="--", linewidth=1.5,
            label=f"Optimal = {optimal_threshold:.2f}")
plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.title("F1 Score vs Classification Threshold")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Summary

| Metric | Default (0.5) | Tuned Threshold |
|---|---|---|
| Accuracy | See output | See output |
| F1 (Churned) | See output | See output |
| AUC | See output | — |

**Architecture:** Input → Dense(128) + BN + Dropout → Dense(64) + BN + Dropout → Dense(32) + Dropout → Sigmoid output.

Class weighting handles the churn imbalance (~14% positive class). Early stopping with `restore_best_weights` prevents overfitting. Threshold tuning shifts the decision boundary to maximize F1 on the minority class.